# Join Resolution: Club Map, Disambiguation, Nicknames, Fuzzy Matching

Phase 1 — Data Inspection, part 4. Notebook 03 proved the join and listed every failure type. This notebook fixes them in order:

1. Club-name map (4 entries: Deportivo, Athletic, Celta, Atlético).
2. Tie-breaking (club-agreed winner first, then most name-words matched, then manual review with birth date and position).
3. Nickname list (Isco, Pepe, Kiko Casilla — verified in the sample).
4. Spelling-tolerant matching (difflib, 0.85 threshold, club agreement still required).
5. Mid-season movers kept as one row per player-team with a split-season flag.

## Cell 1 — Setup and rebuilding the base tables

Rebuilds notebook 03's player-team pairs and cleaned names in condensed form, so this notebook runs standalone from a fresh start.

In [1]:
import json, os, re, unicodedata, difflib
import pandas as pd
import numpy as np

TM_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'transfermarkt'))
SB_DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'statsbomb', 'data'))
SB_COMP, SB_SEASON = 11, 27
SEASON_START, SEASON_END = '2015-07-01', '2016-06-30'

def norm(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode('ascii')
    return re.sub(r'[^a-z ]', '', s.lower()).strip()

ms = json.load(open(os.path.join(SB_DATA_DIR, 'matches', str(SB_COMP), f'{SB_SEASON}.json'), encoding='utf-8'))
sb_rows = []
for m in ms:
    fp = os.path.join(SB_DATA_DIR, 'lineups', f"{m['match_id']}.json")
    for t in json.load(open(fp, encoding='utf-8')):
        for p in t['lineup']:
            sb_rows.append((p['player_name'], t['team_name']))
df_sb = pd.DataFrame(sb_rows, columns=['sb_player', 'sb_team']).drop_duplicates()
df_sb['sb_key'] = df_sb['sb_player'].apply(norm)

app = pd.read_csv(os.path.join(TM_DIR, 'appearances.csv'),
                  usecols=['player_id', 'player_name', 'player_club_id', 'date', 'competition_id'],
                  parse_dates=['date'])
es = app[(app['competition_id'] == 'ES1') & (app['date'] >= SEASON_START) & (app['date'] < '2016-07-01')]
clubs = pd.read_csv(os.path.join(TM_DIR, 'clubs.csv'), usecols=['club_id', 'name'])
df_tm = (es[['player_id', 'player_name', 'player_club_id']].drop_duplicates()
           .merge(clubs, left_on='player_club_id', right_on='club_id')
           .rename(columns={'name': 'tm_club'})[['player_id', 'player_name', 'tm_club']])
df_tm['tm_key'] = df_tm['player_name'].apply(norm)
print('SB pairs:', len(df_sb), '| TM pairs:', len(df_tm))


SB pairs: 612 | TM pairs: 545


## Cell 2 — Club-name map

Four StatsBomb-to-Transfermarkt club translations from notebook 03's mismatch table. The map translates names before the unchanged containment check, so wrong pairs cannot slip in through it; the Deportivo entry alone unblocks 5+ players.

In [2]:
CLUB_MAP = {
    'RC Deportivo La Coru\u00f1a': 'Deportivo de La Coru\u00f1a',
    'Athletic Club': 'Athletic Bilbao',
    'Celta Vigo': 'Celta de Vigo',
    'Atl\u00e9tico Madrid': 'Atl\u00e9tico de Madrid',
}

def club_agree(sb_team, tm_club):
    sb_team = CLUB_MAP.get(sb_team, sb_team)
    return norm(sb_team) in norm(tm_club) or norm(tm_club) in norm(sb_team)

sb_tok = df_sb.assign(tok=df_sb['sb_key'].str.split())
tm_tok = df_tm.assign(tok=df_tm['tm_key'].str.split())
pairs = sb_tok.merge(tm_tok, how='cross', suffixes=('_sb', '_tm'))
pairs = pairs[pairs.apply(lambda r: set(r['tok_tm']).issubset(set(r['tok_sb'])), axis=1)]
pairs['club_ok'] = pairs.apply(lambda r: club_agree(r['sb_team'], r['tm_club']), axis=1)
print('token-subset pairs:', len(pairs), '| with club agreement:', pairs['club_ok'].sum())
print()
print('fixed by the map (sample):')
print(pairs.loc[pairs['sb_team'].isin(CLUB_MAP) & pairs['club_ok'],
      ['sb_player', 'sb_team', 'player_name', 'tm_club']].drop_duplicates().head(8).to_string())


token-subset pairs: 473 | with club agreement: 439

fixed by the map (sample):
                           sb_player                 sb_team        player_name                 tm_club
58967              Celso Borges Mora  RC Deportivo La Coruña       Celso Borges  Deportivo de La Coruña
59492                    Fayçal Fajr  RC Deportivo La Coruña        Fayçal Fajr  Deportivo de La Coruña
60597           Lucas Pérez Martínez  RC Deportivo La Coruña        Lucas Pérez  Deportivo de La Coruña
61135          Pedro Mosquera Parada  RC Deportivo La Coruña     Pedro Mosquera  Deportivo de La Coruña
62222  Sidnei Rechel da Silva Júnior  RC Deportivo La Coruña             Sidnei  Deportivo de La Coruña
62998      Alejandro Arribas Garrido  RC Deportivo La Coruña  Alejandro Arribas  Deportivo de La Coruña
63324    Fernando Navarro i Corbacho  RC Deportivo La Coruña   Fernando Navarro  Deportivo de La Coruña
64087  Luis Alberto Romero Alconchel  RC Deportivo La Coruña       Luis Alberto  Deportiv

## Cell 3 — Tie-breaking rules

Rule 1: exactly one club-agreed candidate wins. Rule 2: most Transfermarkt words matched wins (Rodrigo De Paul's 3 beats Rodrigo's 1). Rule 3: anything left prints with birth date and position for manual review — flagged, never silently dropped.

In [3]:
bio = pd.read_csv(os.path.join(TM_DIR, 'players.csv'),
                  usecols=['player_id', 'date_of_birth', 'position', 'height_in_cm'])
agreed = pairs[pairs['club_ok']].copy()
agreed['n_tok'] = agreed['tok_tm'].apply(len)
agreed = agreed.sort_values(['sb_player', 'n_tok'], ascending=[True, False])
resolved = agreed.drop_duplicates('sb_player')
n_cand = agreed.groupby('sb_player')['player_id'].nunique()
still_multi = n_cand[n_cand > 1]
print('club-agreed SB players:', len(resolved))
print('still ambiguous after cascade:', len(still_multi))
if len(still_multi):
    manual = agreed[agreed['sb_player'].isin(still_multi.index)] \
        [['sb_player', 'sb_team', 'player_id', 'player_name', 'tm_club']] \
        .drop_duplicates().merge(bio, on='player_id', how='left')
    print(manual.to_string())


club-agreed SB players: 433
still ambiguous after cascade: 1
                sb_player   sb_team  player_id      player_name      tm_club        date_of_birth  position  height_in_cm
0  Rodrigo Javier De Paul  Valencia     255901  Rodrigo De Paul  Valencia CF  1994-05-24 00:00:00  Midfield         178.0
1  Rodrigo Javier De Paul  Valencia     131505          Rodrigo  Valencia CF  1991-03-06 00:00:00    Attack         182.0


## Cell 4 — Nickname dictionary

Covers players invisible to word matching (the common name appears nowhere in the legal name) and too different for spelling tolerance. Entries are short, hand-checked, and club-verified when applied.

In [4]:
NICKNAMES = {
    'Francisco Rom\u00e1n Alarc\u00f3n Su\u00e1rez': 'Isco',
    'Kl\u00e9per Laveran Lima Ferreira': 'Pepe',
    'Francisco Casilla Cort\u00e9s': 'Kiko Casilla',
}
nick_rows = []
for sb_name, tm_name in NICKNAMES.items():
    sb_hit = df_sb[df_sb['sb_player'] == sb_name]
    tm_hit = df_tm[df_tm['player_name'] == tm_name]
    if len(sb_hit) and len(tm_hit) and club_agree(sb_hit.iloc[0]['sb_team'], tm_hit.iloc[0]['tm_club']):
        nick_rows.append((sb_hit.iloc[0]['sb_player'], sb_hit.iloc[0]['sb_team'],
                          tm_hit.iloc[0]['player_id'], tm_hit.iloc[0]['player_name'],
                          tm_hit.iloc[0]['tm_club'], 'nickname'))
df_nick = pd.DataFrame(nick_rows,
    columns=['sb_player', 'sb_team', 'player_id', 'player_name', 'tm_club', 'method'])
print('nickname matches (club-verified):', len(df_nick))
print(df_nick.to_string())


nickname matches (club-verified): 3
                        sb_player      sb_team  player_id   player_name      tm_club    method
0  Francisco Román Alarcón Suárez  Real Madrid      85288          Isco  Real Madrid  nickname
1    Kléper Laveran Lima Ferreira  Real Madrid      14132          Pepe  Real Madrid  nickname
2        Francisco Casilla Cortés  Real Madrid      27486  Kiko Casilla  Real Madrid  nickname


## Cell 5 — Token-level fuzzy matching

For still-unmatched players: every Transfermarkt word must score at least 0.85 against its best StatsBomb word (catches casemiro/casimiro at 0.875), and club agreement must still hold. The Llorente/Torres check proves the gate held (Sevilla is not Atlético).

In [5]:
THRESH = 0.85
done_sb = set(resolved['sb_player']) | set(df_nick['sb_player'])
todo_sb = df_sb[~df_sb['sb_player'].isin(done_sb)]
todo_tm = df_tm[~df_tm['player_id'].isin(resolved['player_id'])]
fuzzy_rows = []
for _, s in todo_sb.iterrows():
    for _, t in todo_tm.iterrows():
        scores = [max(difflib.SequenceMatcher(None, tt, st).ratio() for st in s['sb_key'].split())
                  for tt in t['tm_key'].split()]
        if scores and min(scores) >= THRESH and club_agree(s['sb_team'], t['tm_club']):
            fuzzy_rows.append((s['sb_player'], s['sb_team'], t['player_id'],
                               t['player_name'], t['tm_club'], round(min(scores), 3)))
df_fuzzy = pd.DataFrame(fuzzy_rows,
    columns=['sb_player', 'sb_team', 'player_id', 'player_name', 'tm_club', 'min_score'])
print('fuzzy matches:', len(df_fuzzy))
print(df_fuzzy.head(15).to_string())
print()
print('Llorente/Torres audit (must be 0):',
      len(df_fuzzy[df_fuzzy['sb_player'] == 'Fernando Llorente Torres']))


fuzzy matches: 6
                            sb_player        sb_team  player_id      player_name                 tm_club  min_score
0            Carlos Henrique Casimiro    Real Madrid      16306         Casemiro             Real Madrid      0.875
1  José María Martín-Bejarano Serrano     Levante UD     225462        José Mari              Levante UD      0.889
2              Sergio Enrich Ametller          Eibar      81988     Sergi Enrich                SD Eibar      0.909
3               Carlos Gurpegi Nausia  Athletic Club       7631  Carlos Gurpegui         Athletic Bilbao      0.933
4                Robert Ibáñez Castro        Granada     242532     Rober Ibáñez              Granada CF      0.909
5                       Mickaël Ciani       Espanyol      12906    Michaël Ciani  RCD Espanyol Barcelona      0.857

Llorente/Torres audit (must be 0): 0


## Cell 6 — January movers

Players appearing for two clubs in one season (11 here) stay as one row per player-team with a split_season flag. Merging them would mix one club's performances into another's; grouping choices wait for scale-up, made explicitly.

In [6]:
movers = df_sb.groupby('sb_player')['sb_team'].nunique()
movers = movers[movers > 1].index
print('multi-team SB players:', len(movers))
print(df_sb[df_sb['sb_player'].isin(movers)].sort_values('sb_player').to_string())


multi-team SB players: 11
                          sb_player          sb_team                        sb_key
4458       Augusto Matías Fernández       Celta Vigo      augusto matias fernandez
466        Augusto Matías Fernández  Atlético Madrid      augusto matias fernandez
2414                Denis Cheryshev         Valencia               denis cheryshev
3924                Denis Cheryshev      Real Madrid               denis cheryshev
7041         Denis Suárez Fernández          Sevilla        denis suarez fernandez
255          Denis Suárez Fernández       Villarreal        denis suarez fernandez
4082          Francisco Medina Luna          Granada         francisco medina luna
1489          Francisco Medina Luna   Rayo Vallecano         francisco medina luna
6019        Gonzalo Castro Irizábal    Real Sociedad       gonzalo castro irizabal
150         Gonzalo Castro Irizábal           Málaga       gonzalo castro irizabal
500    Guilherme Magdalena Siqueira         Valencia  guilher

## Cell 7 — Final tally and valuation attachment

Every matched row records how it matched (exact/word/nickname/spelling) so accuracy can later be checked per method. Prices attach as latest-on-or-before season end. Unmatched players are listed, not hidden.

In [7]:
exact_hit = df_sb.merge(df_tm, left_on='sb_key', right_on='tm_key', how='inner')
exact_ids = set(exact_hit['sb_player']) & set(resolved['sb_player'])
final = resolved[['sb_player', 'sb_team', 'player_id', 'player_name', 'tm_club']].copy()
final['method'] = final['sb_player'].apply(lambda s: 'exact' if s in exact_ids else 'token')
final = pd.concat([final, df_nick[['sb_player', 'sb_team', 'player_id', 'player_name', 'tm_club', 'method']]],
                  ignore_index=True)
fz = df_fuzzy.drop_duplicates('sb_player')
fz = fz[~fz['sb_player'].isin(final['sb_player'])]
fz['method'] = 'fuzzy'
final = pd.concat([final, fz[['sb_player', 'sb_team', 'player_id', 'player_name', 'tm_club', 'method']]],
                  ignore_index=True)
final['split_season'] = final['sb_player'].isin(
    df_sb.groupby('sb_player')['sb_team'].nunique()[lambda s: s > 1].index)
val = pd.read_csv(os.path.join(TM_DIR, 'player_valuations.csv'), parse_dates=['date'])
val = val[val['date'] <= SEASON_END].sort_values('date').drop_duplicates('player_id', keep='last')
final = final.merge(val[['player_id', 'date', 'market_value_in_eur']], on='player_id', how='left')
print('FINAL matches:', len(final), '/', df_sb['sb_player'].nunique(), 'SB players')
print(final['method'].value_counts().to_dict())
print('with valuation:', final['market_value_in_eur'].notna().sum())
print('split-season rows:', final['split_season'].sum())
print()
print('unmatched SB players:', df_sb['sb_player'].nunique() - final['sb_player'].nunique())
print(final.head(10).to_string())


FINAL matches: 442 / 601 SB players
{'token': 348, 'exact': 85, 'fuzzy': 6, 'nickname': 3}
with valuation: 428
split-season rows: 8

unmatched SB players: 159
                          sb_player         sb_team  player_id          player_name                 tm_club method  split_season       date  market_value_in_eur
0                 Abdoul Karim Yoda          Getafe      61464           Karim Yoda               Getafe CF  token         False 2016-02-22            1800000.0
1                Abdoulaye Doucouré         Granada     127187   Abdoulaye Doucouré              Granada CF  exact         False 2016-02-22            6000000.0
2         Abraham González Casanova        Espanyol      65243     Abraham González  RCD Espanyol Barcelona  token         False 2016-02-22            1000000.0
3       Adalberto Peñaranda Maestre         Granada     308801  Adalberto Peñaranda              Granada CF  token         False 2016-02-22            3000000.0
4  Aderllan Leandro de Jesus Santos 

## Next steps

1. Review the spelling-match table and any manual-review rows.
2. Move the settled logic (cleaning, club map, nicknames, tie-breaking) into src/join/ with tests.
3. Scale to all shared league-seasons.
4. Design Postgres from the player-season-price table.